# Demo 5: Q transfer: the goal heads' query carries the progress

**Insight.** At the pivot, the goal heads' *query* encodes the progress ("these sub-tasks are done"), while the task
span supplies the content. A query taken from another prompt selects whichever sub-task is still pending in the
destination under that progress: neither the source's next sub-task nor the sub-task in the same list position.

**Setup.**
\begin{align*}
p_\mathrm{src}: \mathrm{\underline{a}c\underline{b} \rightarrow c} \\
p_\mathrm{dst}: \mathrm{\underline{a}bd \rightarrow b}
\end{align*}

The source lists $\mathrm{a}, \mathrm{c}, \mathrm{b}$ with $\mathrm{a}$ and $\mathrm{b}$ prefilled, so it writes
$\mathrm{c}$. The destination lists $\mathrm{a}, \mathrm{b}, \mathrm{d}$ with only $\mathrm{a}$ prefilled, so it
writes $\mathrm{b}$. The source's query at the pivot says "$\mathrm{a}$ and $\mathrm{b}$ are done"; read against the
destination's list, the only sub-task left is $\mathrm{d}$. Copying the source's content would give $\mathrm{c}$ (not
even in the destination's list), and copying its list position (2nd) would give $\mathrm{b}$.

**Intervention (gaze-select-restrict).** (1) *Select*: run DST with SRC's query in the 250 goal heads at the pivot
(`" ="`), and pick the DST sub-task the goal heads attend to most. (2) *Restrict*: in a fresh DST run without the query
transfer, from the pivot on, the goal heads may attend only to the selected sub-task, the pivot and the generated
tokens. Every other head sees the whole prompt.

If DST then writes $\mathrm{d}$, the intervention is successful.

**Paper.** §5.1.2 (Figure 6), App. I.2 (Figures 25, 26; Tables 9, 10). This demo runs the instance drawn in Figure 25.

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**The setup** (schematics from the lab talk):

![Query transfer: swap the goal heads' queries, which carry the progress (§5.1.2).](figures/05_q_transfer_setup.png)

*Query transfer: swap the goal heads' queries, which carry the progress (§5.1.2).*

![Step by step: select the pending sub-task from the gaze over the task span, then generate (App. I.2).](figures/05_q_transfer_steps.png)

*Step by step: select the pending sub-task from the gaze over the task span, then generate (App. I.2).*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()      # no generation-config warnings
huggingface_hub.utils.disable_progress_bars()

# as in the paper's figures: source red, destination green, patched run purple
from typing import Literal
ide_theme: Literal['dark', 'light'] = 'dark'
if ide_theme == 'dark':   # light shades; the task-span fill reuses the text color
    SRC_C, DST_C, PAT_C = 'salmon1', 'sea_green3', 'plum2'
    SRC_BG, DST_BG      = SRC_C, DST_C
    PIVOT_BG            = 'bright_yellow'
else:                     # the paper's colors (Figure 5)
    SRC_C, DST_C, PAT_C = '#cc0000', '#38761d', '#5b277d'
    SRC_BG, DST_BG      = '#f4cccc', '#d9ead3'
    PIVOT_BG            = '#ffe6b2'

repo root: /disk/u/arnab/Codes/Projects/goals


In [2]:
CUDA     = '7'
MODEL    = 'Qwen/Qwen2.5-Coder-32B-Instruct'
G250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'
C250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/control/control250__laterhalf.json'
DATATYPE = 'int'
LANG     = 'py-py'                       # SRC-DST languages: py-py, py-cpp, cpp-py, cpp-cpp
VARIANT  = 'order_diff'                  # SRC lists its next sub-task 2nd ('order_matched': 3rd, like DST's)
DONE     = ('is_prime', 'is_positive')   # a, b: the two shared sub-tasks, both done in SRC
SRC_NEXT = 'count_set_bits'              # c: SRC's next sub-task
DST_LAST = 'last_digit'                  # d: DST's 3rd sub-task, the one pending under SRC's progress
ALPHA    = 2.0                           # OV boost of the goal heads, for the patched GoalScope readout only
SEED     = 0

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
Qwen2.5-Coder-32B-Instruct, the 250 goal heads, and 250 random heads (none of them goal heads) as a control.

In [3]:
import hashlib, html as _html
import numpy as np, torch
from rich.console import Console
from rich.table import Table
from rich.text import Text
from rich.markup import escape
from IPython.display import HTML, display

import src.nrgs.data as DATA
import src.nrgs.functional as FUNC
import src.nrgs.eval as E
import scripts.nrgs.q_transfer_eval as QTE
from src.utils.experiment_utils import set_seed

console = Console(width=140, highlight=False)
torch.set_grad_enabled(False)

ctx = FUNC.setup(MODEL, device_map={'': 0})
mt = ctx.mt
goal_heads, ctrl_heads = QTE._resolve_heads(G250), QTE._resolve_heads(C250)
print(f'goal heads: {len(goal_heads)} | control heads: {len(ctrl_heads)} (overlap {len(set(goal_heads) & set(ctrl_heads))})')

2026-11-03 20:56:13 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
goal heads: 250 | control heads: 250 (overlap 0)


## 1. The contrastive pair
The two task lists, what each prompt has already done, and what each writes next on its own.

In [4]:
sl, dl = QTE.LANG_TRANSFERS[LANG]
tag = f"{DATATYPE.replace('[', '(').replace(']', ')')}__{sl}-{dl}"
cfg_seed = (SEED + int(hashlib.md5(f'{VARIANT}/{tag}'.encode()).hexdigest()[:8], 16)) % (2**32)   # the eval's per-cell seed
set_seed(cfg_seed)
setups = DATA.diff_trigger_setups(DATATYPE, variant=VARIANT, n_samples=32, seed=cfg_seed, src_lang=sl, dst_lang=dl)
inst = next(i for i in setups if (tuple(i.src_progress), i.src_target, i.diff_target) == (DONE, SRC_NEXT, DST_LAST))
A, B = inst.src_progress
C, D = inst.src_target, inst.diff_target
LET = {A: 'a', B: 'b', C: 'c', D: 'd'}
COL = {A: 'default', B: DST_C, C: SRC_C, D: PAT_C}

t = Table(title='the two task lists', show_lines=True)
t.add_column(''); t.add_column('SRC'); t.add_column('DST')
fmt = lambda ps: '\n'.join(f'({k}) [{COL[p.id]}]{LET[p.id]}: {escape(p.description)}[/]' for k, p in enumerate(ps, 1))
t.add_row('listed', fmt(inst.src_order), fmt(inst.dst_order))
t.add_row('prefilled (done)', ', '.join(LET[x] for x in inst.src_progress), ', '.join(LET[x] for x in inst.dst_progress))
t.add_row('writes next', f'[{SRC_C}]c[/]', f'[{DST_C}]b[/]')
console.print(t)
console.print(f'patched DST: progress -> [{PAT_C}]d[/]  |  literal copy -> [{SRC_C}]c[/]  |  positional copy / no effect -> [{DST_C}]b[/]')

                                                   the two task lists                                                    
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                  ┃ SRC                                                        ┃ DST                                   ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ listed           │ (1) a: whether the number is prime                         │ (1) a: whether the number is prime    │
│                  │ (2) c: the number of set bits in the binary representation │ (2) b: whether the number is positive │
│                  │ (3) b: whether the number is positive                      │ (3) d: the last decimal digit         │
├──────────────────┼────────────────────────────────────────────────────────────┼───────────────────────────────────────┤
│ prefilled (done) │ a, b                                                       │ a                                     │
├──────────────────┼────────────────────────────────────────────────────────────┼───────────────────────────────────────┤
│ writes next      │ c                                                          │ b                                     │
└──────────────────┴────────────────────────────────────────────────────────────┴───────────────────────────────────────┘

patched DST: progress -> d  |  literal copy -> c  |  positional copy / no effect -> b

The two prompts as the model sees them. Both end at the pivot ` =` of the next line; that is where SRC's query is
read and where it is written into DST. The task lists are highlighted.

In [5]:
mat = DATA.materialize(inst)

def show(prompt, preds, title, bg):
    ids = FUNC.prepare_input(prompts=prompt, tokenizer=mt)['input_ids'][0].tolist()
    spans = [FUNC.token_span(prompt, p.description) for p in preds]
    win = set(range(min(s[0] for s in spans), max(s[1] for s in spans)))
    start = max(i for i, t in enumerate(ids) if t == mt.tokenizer.convert_tokens_to_ids('<|im_start|>') and i < min(win))
    txt = Text()
    for i in range(start, len(ids)):
        s = mt.tokenizer.decode([ids[i]])
        pivot = i == len(ids) - 1                     # the pivot: the prompt's last token, ` =`
        txt.append(s, style=f'bold black on {PIVOT_BG}' if pivot else f'black on {bg}' if i in win else 'dim')
    console.rule(title); console.print(txt)

show(mat['src_prompt'], inst.src_order, 'SRC prompt (query donor)', SRC_BG)
show(mat['dst_prompt'], inst.dst_order, 'DST prompt (the run we patch)', DST_BG)

───────────────────────────────────────────────────────── SRC prompt (query donor) ─────────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes:
(1) whether the number is prime
(2) the number of set bits in the binary representation
(3) whether the number is positive
<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = n > 1 and all(n % i != 0 for i in range(2, int(n**0.5) + 1))
    b = (n > 0)
    c =

────────────────────────────────────────────────────── DST prompt (the run we patch) ───────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes:
(1) whether the number is prime
(2) whether the number is positive
(3) the last decimal digit
<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = n > 1 and all(n % i != 0 for i in range(2, int(n**0.5) + 1))
    b =

## 2. Clean runs
One call runs the whole procedure (clean runs, selection, restricted run, readouts) for the goal heads, and one for the
control heads; we look at its parts in turn. Without any intervention, SRC writes c and DST writes b, and GoalScope at
the pivot verbalizes what the goal heads encode there.

In [6]:
stub = lambda prompt: prompt.rsplit('\n', 1)[-1]          # the prefilled frontier stub, e.g. `    b =`

def show_run(title, color, prompt, gen, readout=None, verdict=None):
    console.rule(f'[bold {color}]{title}[/]')
    if readout is not None:
        console.print(f'[bold]GoalScope at the pivot:[/] [italic]{escape(readout.strip())}[/]')
    console.print(f'[bold]continuation:[/]\n[grey50]{escape(stub(prompt))}[/][{color}]{escape(gen.rstrip())}[/]')
    if verdict: console.print(f'[bold]judge:[/] {verdict}')
    console.print()

with FUNC.console.capture():                               # silence the per-readout log lines
    raw = FUNC.run_q_transfer_intervention(mt, inst, goal_heads, readout_heads=goal_heads, max_new=QTE.MAX_NEW,
                                          alpha=ALPHA, compute_readout=True, gsr_max_new=QTE.GSR_MAX_NEW)
    raw_ctrl = FUNC.run_q_transfer_intervention(mt, inst, ctrl_heads, readout_heads=ctrl_heads, max_new=QTE.MAX_NEW,
                                               alpha=ALPHA, compute_readout=False, gsr_max_new=QTE.GSR_MAX_NEW)
show_run('SRC run', SRC_C, raw['src_prompt'], raw['gen_src'], raw['ro_src'])
show_run('DST run', DST_C, raw['dst_prompt'], raw['gen_dst'], raw['ro_dst'])

───────────────────────────────────────────────────────────────── SRC run ──────────────────────────────────────────────────────────────────

GoalScope at the pivot: I'm checking if a number is positive and counting the number of 1s in its binary representation.

continuation:
    c = bin(n).count('1')
    return a, c, b
</code>

───────────────────────────────────────────────────────────────── DST run ──────────────────────────────────────────────────────────────────

GoalScope at the pivot: I'm checking if the number is greater than zero.

continuation:
    b = n > 0
    c = n % 10
    return a, b, c
</code>

## 3. Select: where the pivot looks
The goal heads' attention (value-weighted, mean over the heads) from DST's pivot onto each listed sub-task: shading
per token, and the mean per sub-task on the right. Left: the clean DST pivot. Right: the same pivot with SRC's query.
The sub-task with the highest mean under SRC's query is selected. Below, the same for the 250 random heads.

In [7]:
def gaze_panels(raw, inst, heads_name):
    dst = raw['dst_prompt']
    pos = raw['task_positions']
    att = {'clean': dict(zip(pos, raw['attn_clean'])), 'patched': dict(zip(pos, raw['attn_patched']))}
    items = {p.id: list(range(*FUNC.token_span(dst, p.description))) for p in inst.task.predicates}
    score = {m: {pid: float(np.mean([att[m][q] for q in ps])) for pid, ps in items.items()} for m in att}
    amax = max(att[m][q] for m in att for ps in items.values() for q in ps)
    gmax = max(max(s.values()) for s in score.values())
    ids = FUNC.prepare_input(prompts=dst, tokenizer=mt)['input_ids'][0].tolist()
    tok = lambda q: _html.escape(mt.tokenizer.decode([ids[q]]))
    def panel(m, title):
        best = max(score[m], key=score[m].get)
        rows = ''
        for k, p in enumerate(inst.task.predicates, 1):
            spans = ''.join(f'<span style="background:rgba(63,143,90,{0.04 + 0.66 * min(1, att[m][q] / amax):.3f});'
                            f'border-radius:2px">{tok(q)}</span>' for q in items[p.id])
            g = score[m][p.id]
            bar = (f'<span style="display:inline-block;width:{110 * g / gmax:.0f}px;height:11px;background:#3f8f5a;'
                   f'opacity:{0.35 + 0.65 * g / gmax:.2f};border-radius:2px"></span> '
                   f'<span style="font-weight:{600 if p.id == best else 400}">{g:.3f}</span>')
            rows += (f'<tr><td style="color:#d23c35;font-family:serif;font-size:16px;padding-right:6px">{LET[p.id]}</td>'
                     f'<td style="font-family:monospace;white-space:pre">({k}){spans}</td>'
                     f'<td style="padding-left:14px;text-align:left">{bar}</td></tr>')
        return (f'<div style="flex:1;background:#f7f6f3;padding:10px 12px;border-radius:6px;color:#1f1e1c">'
                f'<div style="font-weight:600;margin-bottom:6px">{title}</div><table>{rows}</table></div>')
    display(HTML(f'<div style="display:flex;gap:14px">'
                 f'{panel("clean", f"(ii) clean DST pivot · {heads_name}")}'
                 f'{panel("patched", f"(iii) selection pass: SRC query at the DST pivot · {heads_name}")}</div>'))
    return score

for name, r in (('goal 250', raw), ('control 250', raw_ctrl)):
    gaze_panels(r, inst, name)
    console.print(f'{name}: selected [bold]{LET[r["cl_selected"]]}[/] ({escape(r["cl_selected"])})\n')

a,(1) whether the number is prime,0.065
b,(2) whether the number is positive,0.239
d,(3) the last decimal digit,0.062
a,(1) whether the number is prime,0.051
b,(2) whether the number is positive,0.074
d,(3) the last decimal digit,0.306


goal 250: selected d (last_digit)

a,(1) whether the number is prime,0.041
b,(2) whether the number is positive,0.055
d,(3) the last decimal digit,0.026
a,(1) whether the number is prime,0.018
b,(2) whether the number is positive,0.021
d,(3) the last decimal digit,0.054


control 250: selected d (last_digit)

## 4. Restrict and generate
From the pivot on, the goal heads may read only the selected sub-task's description, the pivot and the generated
tokens (highlighted below); every other head still sees the whole prompt. The control restricts the 250 random heads to
the sub-task *they* selected.

For each run we show the GoalScope readout at the pivot under SRC's query (goal heads only), the generation, and the
verdict of an LM judge (Claude): which sub-task does the code's first computation implement (d is a **hit**), and did
the readout move from b to d?

In [8]:
ids = FUNC.prepare_input(prompts=raw['dst_prompt'], tokenizer=mt)['input_ids'][0].tolist()
keep = set(raw['cl_restrict_keep'])
start = max(i for i, t in enumerate(ids) if t == mt.tokenizer.convert_tokens_to_ids('<|im_start|>') and i < min(keep))
txt = Text()
for i in range(start, len(ids)):
    pivot = i == len(ids) - 1
    txt.append(mt.tokenizer.decode([ids[i]]), style=f'bold black on {PIVOT_BG}' if pivot else f'black on {DST_BG}' if i in keep else 'dim')
console.rule('what the goal heads can read from the DST prompt'); console.print(txt); console.print()

JL = {'native': f'[bold {DST_C}]b (DST\'s own)[/]', 'src_target': f'[bold {SRC_C}]c (SRC\'s next)[/]',
      'diff_target': f'[bold {PAT_C}]d (pending)[/]', 'neither': 'neither'}
RL = {'src': f'[bold {PAT_C}]d[/]', 'dst': f'[bold {DST_C}]b[/]'}
for name, r in (('goal 250', raw), ('control 250', raw_ctrl)):
    causal = E.judge_causal(inst, r['gen_patched'], model_name=QTE.ORACLE_MODEL)
    joint = E.judge_joint(inst, r['gen_patched'], model_name=QTE.ORACLE_MODEL)
    verdict = (f"code implements {JL.get(joint['label'], joint['label'])} "
               f"({'[bold]HIT[/]' if causal['triggered'] else '[dim]miss[/]'})")
    if r['ro_patched'] is not None:
        ro = E.eval_readouts(inst, r['ro_src'], r['ro_dst'], r['ro_patched'], model_name=QTE.ORACLE_MODEL)
        verdict += f"; readout says {RL.get(ro['patched'], ro['patched'])}{' ([bold]shifted[/])' if ro['shifted'] else ''}"
    show_run(f'GSR DST run: {name} (selected {LET[r["cl_selected"]]})', PAT_C, r['dst_prompt'], r['gen_patched'],
             r['ro_patched'], verdict)

───────────────────────────────────────────── what the goal heads can read from the DST prompt ─────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes:
(1) whether the number is prime
(2) whether the number is positive
(3) the last decimal digit
<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = n > 1 and all(n % i != 0 for i in range(2, int(n**0.5) + 1))
    b =

──────────────────────────────────────────────────── GSR DST run: goal 250 (selected d) ────────────────────────────────────────────────────

GoalScope at the pivot: I'm checking the last digit of the number by using the modulus operator (`% 10`).

continuation:
    b = n % 10
    return b

# Example usage:
# print(func(123))  # Output: 3
</code>

judge: code implements d (pending) (HIT); readout says d (shifted)

────────────────────────────────────────────────── GSR DST run: control 250 (selected d) ───────────────────────────────────────────────────

continuation:
    b = n > 0
    c = n % 10
    return a, b, c
</code>

judge: code implements b (DST's own) (miss)

## 5. Does the restricted run's code work?
As in Demo 4, `src.nrgs.exec_check` compiles the generated function and runs its frontier statement on test inputs
against the reference implementation of each candidate sub-task: b (DST's own next), c (SRC's next, a literal copy)
and d (pending under SRC's progress). **complete**: the same value on every test input; **complete on typical inputs**:
on non-negative ints and non-empty strings / lists. The check isolates the frontier statement, so it does not depend
on how the generation orders its `return`.

In [ ]:
from src.nrgs import exec_check as XC

targets = [B, C, D]
listed = [p.id for p in inst.dst_order]                           # test inputs must suit every listed sub-task
cell = lambda v: (f"{v['n_pass']}/{v['n']}" + (' [bold]complete[/]' if v['complete']
                  else ' complete on typical inputs' if v['complete_typical'] else ''))
t = Table(title='execution check: frontier statement vs the reference implementations')
t.add_column('run'); t.add_column('compiles')
for x in targets:
    t.add_column(f'[{COL[x]}]{LET[x]}: {escape(x)}[/]')
runs = [('DST run (clean)', raw['gen_dst']), ('GSR: goal 250', raw['gen_patched']), ('GSR: control 250', raw_ctrl['gen_patched'])]
for name, gen in runs:
    r = XC.check(raw['dst_prompt'], gen, datatype=DATATYPE, targets=targets, listed=listed)
    t.add_row(name, 'yes' if r['compiles'] else f"[red]no[/] {escape(r['compile_error'][:60])}",
              *[cell(r['targets'][x]) for x in targets])
console.print(t)

## What to expect
<img src="figures/05_q_transfer_results.png" width="400" alt="Q transfer (gaze-select-restrict) results by language direction, order-diff (Figure 6, right, in the paper)">

Through the goal heads, the restricted DST run should write d, the sub-task still pending under SRC's progress
(causal-shift), and the readout at the pivot should move from b to d (readout-shift). Across 512 order-diff instances
this holds 57% of the time for the code (51–66% per language direction) and 41% for the readout; when SRC lists its
next sub-task in the same slot as DST's (order-matched), 87%. The same procedure through 250 random heads works on 3–4%.
See §5.1.2 and Figure 6 in the paper; App. I.2 (Figures 25, 26, Tables 9, 10) breaks it down by datatype.